# 03 — Double factorization with PyLIQTR and Qualtran

Two-electron integrals are decomposed as
$$h_{pqrs} = \sum_{\ell=1}^{L} g_\ell^{(1)} L_{pq}^{(\ell)} L_{rs}^{(\ell)},$$
where each factor matrix $L^{(\ell)}$ is itself diagonalized to give a second
factorization layer. Block-encoding cost drops from $\mathcal{O}(N^4)$ in the
plain LCU to $\mathcal{O}(N \Xi)$ with $\Xi$ the truncated rank.

This notebook tries three approaches:

1. **Qualtran** — `DoubleFactorizationBlockEncoding` bloq with built-in
   resource counting (`get_cost_value` + `QECGatesCost`).
2. **PyLIQTR** — `getEncoding(VALID_ENCODINGS.DoubleFactorized)` paired with
   `QubitizedPhaseEstimation` and `estimate_resources`.
3. **Analytical fallback** — eigendecomposition of the symmetrized two-electron
   tensor and Lee/Babbush-style Toffoli costing.

If the library imports fail or the API has shifted, the notebook still produces
a complete result table from the analytical path.

In [13]:
import json
import math
import os
import sys
import numpy as np
from openfermion import InteractionOperator, jordan_wigner

sys.path.insert(0, ".")
from config import load_config, ensure_dirs
cfg = load_config()
ensure_dirs(cfg)

npz_path = os.path.join(cfg["paths"]["data_dir"], "integrals.npz")
npz = np.load(npz_path)
labels = [a["label"] for a in cfg["active_spaces"]]
all_data = {}
for label in labels:
    all_data[label] = {
        "nelec": int(npz[f"{label}_nelec"]),
        "norb": int(npz[f"{label}_norb"]),
        "e_core": float(npz[f"{label}_e_core"]),
        "h1e_eff": npz[f"{label}_h1e_eff"],
        "h2e_phys": npz[f"{label}_h2e_phys"],
    }
epsilon = cfg["qpe"]["epsilon_ha"]

## Path 1 — Qualtran

`qualtran.bloqs.chemistry.df.double_factorization.DoubleFactorizationBlockEncoding`
gives a Toffoli-counted block encoding. Wrap in qubitized QPE and call
`get_cost_value(bloq, QECGatesCost())` to extract T- and Toffoli counts.

In [14]:
qualtran_results = {}
qualtran_available = False

try:
    from qualtran.bloqs.chemistry.df.double_factorization import (
        DoubleFactorizationBlockEncoding,
    )
    from qualtran.resource_counting import get_cost_value, QECGatesCost
    qualtran_available = True
    print("qualtran imported successfully")
except Exception as e:
    print(f"qualtran unavailable: {e}")

if qualtran_available:
    for label in labels:
        d = all_data[label]
        norb = d["norb"]
        h2e = d["h2e_phys"]

        # First factorization: eigendecomp of (pq|rs) viewed as a matrix in (pq, rs)
        h2e_flat = h2e.reshape(norb**2, norb**2)
        h2e_sym = 0.5 * (h2e_flat + h2e_flat.T)
        eigvals = np.linalg.eigvalsh(h2e_sym)
        eigvals = eigvals[np.abs(eigvals) > 1e-6]
        L_outer = len(eigvals)            # outer (auxiliary) rank
        L_inner = norb                    # inner rank (eigenmodes per factor)

        try:
            bloq = DoubleFactorizationBlockEncoding(
                num_spin_orb=2 * norb,
                num_aux=L_outer,
                num_eig=L_outer * L_inner,
            )
            cost = get_cost_value(bloq, QECGatesCost())
            t_per_walk = int(cost.t + 4 * cost.toffoli)
            qualtran_results[label] = {
                "L_outer": int(L_outer),
                "L_inner": int(L_inner),
                "t_per_walk_qualtran": t_per_walk,
                "toffoli_per_walk": int(cost.toffoli),
            }
            print(f"({d['nelec']}e,{norb}o): qualtran T/walk = {t_per_walk:,}, "
                  f"Toffoli/walk = {int(cost.toffoli):,}")
        except Exception as e:
            qualtran_results[label] = {"error": str(e)}
            print(f"({d['nelec']}e,{norb}o): qualtran failed -- {e}")

qualtran unavailable: No module named 'qualtran'


## Path 2 — PyLIQTR

`pyLIQTR.BlockEncodings.getEncoding(VALID_ENCODINGS.DoubleFactorized)` produces
a Cirq-level block encoding circuit. Wrap with `QubitizedPhaseEstimation` and
call `estimate_resources` for T-counts.

In [15]:
pyliqtr_results = {}
pyliqtr_available = False

try:
    from pyLIQTR.ProblemInstances.ChemicalHamiltonian import ChemicalHamiltonian
    from pyLIQTR.BlockEncodings.getEncoding import getEncoding, VALID_ENCODINGS
    from pyLIQTR.qubitization.phase_estimation import QubitizedPhaseEstimation
    from pyLIQTR.utils.resource_analysis import estimate_resources
    pyliqtr_available = True
    print("pyLIQTR imported successfully")
except Exception as e:
    print(f"pyLIQTR unavailable: {e}")

if pyliqtr_available:
    for label in labels:
        d = all_data[label]
        norb = d["norb"]
        try:
            chem = ChemicalHamiltonian(
                h1=d["h1e_eff"],
                eri=d["h2e_phys"],
                core_energy=d["e_core"],
            )
            enc = getEncoding(VALID_ENCODINGS.DoubleFactorized)(chem)
            qpe = QubitizedPhaseEstimation(block_encoding=enc, eps=epsilon)
            res = estimate_resources(qpe.circuit)
            pyliqtr_results[label] = {
                "T": int(res.get("T", 0) or res.get("t", 0)),
                "logical_qubits": int(res.get("LogicalQubits",
                                              res.get("Qubits", 0))),
            }
            print(f"({d['nelec']}e,{norb}o): pyLIQTR T = "
                  f"{pyliqtr_results[label]['T']:,}, "
                  f"Q_L = {pyliqtr_results[label]['logical_qubits']}")
        except Exception as e:
            pyliqtr_results[label] = {"error": str(e)}
            print(f"({d['nelec']}e,{norb}o): pyLIQTR failed -- {e}")

pyLIQTR unavailable: No module named 'pyLIQTR'


## Path 3 — Analytical DF (always runs)

Eigendecompose the symmetrized two-electron matrix; sum eigenvalue magnitudes
for the 2e contribution to lambda; combine with $\lambda_{1e} = \sum_{pq}|h_{pq}^{\text{eff}}|$.

Walk-operator cost follows the Lee/Babbush DF formula:
$$N_T^{\text{walk}} \approx 4 \bigl[N \Xi + \Xi \lceil \log_2 \Xi \rceil + 2\bigr] \cdot 4$$
(the trailing factor of 4 converts Toffoli to T-gates).

In [16]:
def double_factorize(h1e, h2e, norb, threshold=1e-6):
    h2e_flat = h2e.reshape(norb**2, norb**2)
    h2e_sym = 0.5 * (h2e_flat + h2e_flat.T)
    eigvals, eigvecs = np.linalg.eigh(h2e_sym)
    mask = np.abs(eigvals) > threshold
    eigvals = eigvals[mask]
    eigvecs = eigvecs[:, mask]
    rank = len(eigvals)
    return {
        "lambda_1e": float(np.sum(np.abs(h1e))),
        "lambda_2e": float(np.sum(np.abs(eigvals))),
        "rank": int(rank),
        "eigvals": eigvals,
    }


df_analytical = {}
for label in labels:
    d = all_data[label]
    norb = d["norb"]
    df = double_factorize(d["h1e_eff"], d["h2e_phys"], norb)

    lam_df = df["lambda_1e"] + df["lambda_2e"]
    rank = df["rank"]
    log2_rank = max(1, math.ceil(math.log2(rank)))

    # Lee/Babbush-style walk cost
    toff_per_walk = 2 * (norb * rank + rank * log2_rank + 2)
    t_per_walk = 4 * toff_per_walk

    prec = int(np.ceil(np.log2(lam_df / epsilon))) + 1
    qpe_rounds = 2 ** prec
    t_total = qpe_rounds * t_per_walk

    ancilla = log2_rank + math.ceil(math.log2(norb)) + prec
    logical_qubits = 2 * norb + ancilla + 1

    df_analytical[label] = {
        "active_space": f"({d['nelec']}e, {norb}o)",
        "lambda_1e": df["lambda_1e"],
        "lambda_2e": df["lambda_2e"],
        "lambda_total": float(lam_df),
        "df_rank": rank,
        "qpe_bits": int(prec),
        "qpe_rounds": int(qpe_rounds),
        "t_per_walk": int(t_per_walk),
        "t_total": int(t_total),
        "logical_qubits": int(logical_qubits),
    }

    print(f"({d['nelec']}e,{norb}o): rank={rank:>4}  lambda={lam_df:>7.2f}  "
          f"QPE={prec:>2}  T={t_total:>16,}  Q_L={logical_qubits}")

(4e,4o): rank=  16  lambda=   7.86  QPE=14  T=      17,039,360  Q_L=29
(8e,8o): rank=  64  lambda=  34.85  QPE=16  T=     470,810,624  Q_L=42
(12e,12o): rank= 144  lambda=  79.62  QPE=17  T=   3,021,996,032  Q_L=54
(16e,16o): rank= 256  lambda= 142.34  QPE=18  T=  12,889,096,192  Q_L=63
(20e,20o): rank= 400  lambda= 229.15  QPE=19  T=  48,662,315,008  Q_L=74
(24e,24o): rank= 576  lambda= 339.60  QPE=19  T=  82,149,638,144  Q_L=83


## Compare the three paths

In [17]:
print(f"{'Space':>10} {'Method':>12} {'lambda':>10} {'T-gates':>20} {'Q_L':>6}")
for label in labels:
    r = df_analytical[label]
    print(f"{r['active_space']:>10} {'Analytical':>12} "
          f"{r['lambda_total']:>10.2f} {r['t_total']:>20,} "
          f"{r['logical_qubits']:>6}")
    if label in qualtran_results and "error" not in qualtran_results[label]:
        q = qualtran_results[label]
        # Combine qualtran t_per_walk with our QPE rounds
        t_total_q = r["qpe_rounds"] * q["t_per_walk_qualtran"]
        print(f"{'':>10} {'Qualtran':>12} {'':>10} {t_total_q:>20,} {'':>6}")
    if label in pyliqtr_results and "error" not in pyliqtr_results[label]:
        p = pyliqtr_results[label]
        print(f"{'':>10} {'PyLIQTR':>12} {'':>10} {p['T']:>20,} "
              f"{p['logical_qubits']:>6}")

     Space       Method     lambda              T-gates    Q_L
  (4e, 4o)   Analytical       7.86           17,039,360     29
  (8e, 8o)   Analytical      34.85          470,810,624     42
(12e, 12o)   Analytical      79.62        3,021,996,032     54
(16e, 16o)   Analytical     142.34       12,889,096,192     63
(20e, 20o)   Analytical     229.15       48,662,315,008     74
(24e, 24o)   Analytical     339.60       82,149,638,144     83


## Save DF results

In [18]:
out = {
    "analytical": df_analytical,
    "qualtran": qualtran_results if qualtran_available else "unavailable",
    "pyliqtr": pyliqtr_results if pyliqtr_available else "unavailable",
}
out_path = os.path.join(cfg["paths"]["data_dir"], "df_results.json")
with open(out_path, "w") as f:
    json.dump(out, f, indent=2, default=str)
print(f"saved {out_path}")

saved data/df_results.json
